In [3]:
%pip install pyzmq

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\rishit\Human-Activity-Recognition\env\Scripts\python.exe -m pip install --upgrade pip' command.


In [1]:
%pip install jupyter 

  Using cached argon2_cffi-23.1.0-py3-none-any.whl (15 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an EnvironmentError: [Errno 2] No such file or directory: 'C:\\Users\\HP\\AppData\\Local\\Temp\\pip-install-u2gvtx_9\\jupyterlab-widgets\\jupyterlab_widgets-3.0.16.data/data/share/jupyter/labextensions/@jupyter-widgets/jupyterlab-manager/static/packages_base_lib_index_js-webpack_sharing_consume_default_jquery_jquery.5dd13f8e980fa3c50bfe.js'

You should consider upgrading via the 'c:\rishit\Human-Activity-Recognition\env\Scripts\python.exe -m pip install --upgrade pip' command.


In [8]:
%pip install numpy pandas scikit-learn matplotlib seaborn torch torchvision

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\rishit\Human-Activity-Recognition\env\Scripts\python.exe -m pip install --upgrade pip' command.


In [2]:
"""
MS-GRS-BiLSTM for Person recognition from Kinect skeletal CSV.
Saves best model to best_model.pt and prints confusion matrix, classification report, test accuracy.

Usage:
    python ms_grs_bilstm_person_recognition.py --data_path /path/to/combined_kinect_dataset.csv
"""

import argparse
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# -------------------------
# Utilities / DatasetPrep
# -------------------------
def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

class KinectSequenceDataset(Dataset):
    """
    Dataset built from sliding windows of the skeleton features.
    Each window labeled by Person (string).
    Expects a pd.DataFrame with columns including 'timestamp', 'Person', 'Activity' and joint numeric columns.
    Windows are generated per (Person, Activity) group in time order so sequence boundaries match activities.
    """
    def __init__(self, df, feature_cols, seq_len=64, stride=16, scaler=None, label_encoder=None):
        self.seq_len = seq_len
        self.stride = stride
        self.feature_cols = feature_cols
        self.scaler = scaler
        self.label_encoder = label_encoder

        sequences = []
        labels = []

        # Group by Person+Activity to generate consistent windows per activity segment
        grouped = df.groupby(['Person', 'Activity'])
        for (person, activity), g in grouped:
            g_sorted = g.sort_values('timestamp')
            X = g_sorted[feature_cols].values.astype(np.float32)
            if scaler is not None:
                X = scaler.transform(X)
            n = len(X)
            # sliding windows
            if n >= seq_len:
                for start in range(0, n - seq_len + 1, stride):
                    sequences.append(X[start:start + seq_len])
                    labels.append(person)
            else:
                # pad short segments (repeat last row) to create one window
                pad_needed = seq_len - n
                Xpad = np.vstack([X, np.repeat(X[-1][None, :], pad_needed, axis=0)])
                sequences.append(Xpad)
                labels.append(person)

        self.X = np.array(sequences)  # shape (N, seq_len, n_features)
        self.y = np.array(labels)
        assert len(self.X) == len(self.y)

        if self.label_encoder is not None:
            self.y_enc = self.label_encoder.transform(self.y)
        else:
            self.y_enc = None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx]).float()  # (seq_len, n_features)
        if self.y_enc is None:
            y = self.y[idx]
        else:
            y = torch.tensor(self.y_enc[idx]).long()
        return x, y

# -------------------------
# Model: Multiscale GRS-BiLSTM
# -------------------------
class GatedResidualBiLSTMBlock(nn.Module):
    """
    One BiLSTM block with residual connection + gating.
    Input: (batch, seq_len, feature_dim)
    Output: (batch, seq_len, hidden*2)  (bidirectional)
    We'll apply a linear projection to maintain dimension if needed.
    """
    def __init__(self, input_dim, hidden_dim, dropout=0.2):
        super().__init__()
        self.bilstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        # project back to input_dim for residual addition (if dims differ)
        self.project = nn.Linear(hidden_dim * 2, input_dim) if (hidden_dim * 2) != input_dim else nn.Identity()
        # gating: sigmoid gate over projected output
        self.gate = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.Sigmoid()
        )
        self.layer_norm = nn.LayerNorm(input_dim)

    def forward(self, x):
        # x: (B, T, D)
        out, _ = self.bilstm(x)  # (B, T, 2H)
        out = self.dropout(out)
        proj = self.project(out)  # (B, T, D') -> D'
        gate = self.gate(proj)  # (B, T, D')
        # gated residual: x + gate * proj
        res = x + gate * proj
        res = self.layer_norm(res)
        return res  # (B, T, D')

class MultiScaleGRSBiLSTM(nn.Module):
    """
    Multiscale model: create multiple temporal scales of input by simple average pooling/downsampling.
    For each scale, a stack of GatedResidualBiLSTMBlock is applied. Outputs are pooled (global avg) and concatenated.
    Final classifier MLP predicts Person classes.
    """
    def __init__(self, input_dim, hidden_dim, n_scales=3, n_blocks_per_scale=2, num_classes=10, dropout=0.3):
        super().__init__()
        self.n_scales = n_scales
        self.hidden_dim = hidden_dim
        self.blocks = nn.ModuleList()
        for _ in range(n_scales):
            # each scale uses the same block architecture
            scale_blocks = nn.ModuleList([GatedResidualBiLSTMBlock(input_dim, hidden_dim, dropout=dropout)
                                          for _ in range(n_blocks_per_scale)])
            self.blocks.append(scale_blocks)

        # after pooling each branch -> representation size input_dim
        concatenated_dim = input_dim * n_scales
        self.classifier = nn.Sequential(
            nn.Linear(concatenated_dim, concatenated_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(concatenated_dim // 2, num_classes)
        )

    def forward(self, x):
        # x: (B, T, D)
        B, T, D = x.shape
        branch_outs = []
        for s in range(self.n_scales):
            # downsample factor = 2^s (scale 0 -> factor1)
            factor = 2 ** s
            if factor > 1:
                # simple downsample by average pooling across temporal windows
                # reshape to (B, new_T, factor, D) and mean over factor
                new_T = T // factor
                if new_T < 1:
                    # if factor too big, fallback to simple mean over time -> (B,1,D)
                    xs = x.mean(dim=1, keepdim=True)
                else:
                    truncated = x[:, :new_T * factor, :].reshape(B, new_T, factor, D)
                    xs = truncated.mean(dim=2)  # (B, new_T, D)
            else:
                xs = x  # scale 0 is original

            # pass through stack of blocks
            for block in self.blocks[s]:
                xs = block(xs)  # (B, T_s, D')
            # pool over time -> (B, D')
            pooled = xs.mean(dim=1)
            branch_outs.append(pooled)

        concat = torch.cat(branch_outs, dim=1)  # (B, D'*n_scales)
        logits = self.classifier(concat)
        return logits

# -------------------------
# Training / Evaluation
# -------------------------
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_targets = []
    for X, y in loader:
        X = X.to(device)  # (B, T, D)
        y = y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X.size(0)
        preds = logits.argmax(dim=1).detach().cpu().numpy()
        all_preds.extend(preds.tolist())
        all_targets.extend(y.detach().cpu().numpy().tolist())
    avg_loss = total_loss / len(loader.dataset)
    acc = (np.array(all_preds) == np.array(all_targets)).mean()
    return avg_loss, acc

def eval_model(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            y = y.to(device)
            logits = model(X)
            loss = criterion(logits, y)
            total_loss += loss.item() * X.size(0)
            preds = logits.argmax(dim=1).detach().cpu().numpy()
            all_preds.extend(preds.tolist())
            all_targets.extend(y.detach().cpu().numpy().tolist())
    avg_loss = total_loss / len(loader.dataset)
    acc = (np.array(all_preds) == np.array(all_targets)).mean()
    return avg_loss, acc, np.array(all_targets), np.array(all_preds)

# -------------------------
# Main pipeline
# -------------------------
def main(args):
    set_seed(args.seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print("Device:", device)

    # Read CSV
    df = pd.read_csv(args.data_path)
    print("Loaded dataframe with shape:", df.shape)

    # columns: drop body_id? keep timestamp for ordering.
    # Identify feature columns: all numeric joint columns (exclude timestamp, Person, Activity, body_id optionally)
    exclude = {'timestamp', 'Person', 'Activity', 'body_id'}
    feature_cols = [c for c in df.columns if c not in exclude]
    print("Detected feature columns count:", len(feature_cols))

    # Basic cleaning: drop rows with NaNs (alternatively you could interpolate)
    before = len(df)
    df = df.dropna(subset=feature_cols + ['Person', 'Activity', 'timestamp'])
    after = len(df)
    if before != after:
        print(f"Dropped {before-after} rows with NaNs.")

    # Label encode Person
    le = LabelEncoder()
    df['Person'] = df['Person'].astype(str)
    le.fit(df['Person'].values)
    num_classes = len(le.classes_)
    print("Persons (classes):", num_classes)
    print("Class distribution:", Counter(df['Person'].values))

    # Feature scaling: fit scaler on whole dataset features
    scaler = StandardScaler()
    scaler.fit(df[feature_cols].values.astype(np.float32))

    # Create sequences dataset (this generates windows across all Person+Activity groups)
    dataset = KinectSequenceDataset(df, feature_cols, seq_len=args.seq_len, stride=args.stride,
                                    scaler=scaler, label_encoder=le)
    # split train/val/test by stratified sampling on person labels at window level (simple)
    X_idx = np.arange(len(dataset))
    y_enc = dataset.y_enc
    idx_train, idx_temp, y_train, y_temp = train_test_split(X_idx, y_enc, test_size=args.test_val_split, 
                                                            stratify=y_enc, random_state=args.seed)
    idx_val, idx_test, y_val, y_test = train_test_split(idx_temp, y_temp, test_size=args.test_ratio_of_temp,
                                                        stratify=y_temp, random_state=args.seed)
    def subset_from_indices(ds, indices):
        # create a small dataset-like object for DataLoader using torch.utils.data.Subset
        from torch.utils.data import Subset
        return Subset(ds, indices)

    train_ds = subset_from_indices(dataset, idx_train)
    val_ds = subset_from_indices(dataset, idx_val)
    test_ds = subset_from_indices(dataset, idx_test)

    print(f"Windows: train {len(train_ds)}, val {len(val_ds)}, test {len(test_ds)}")

    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=args.batch_size, shuffle=False)

    # Build model
    input_dim = len(feature_cols)
    model = MultiScaleGRSBiLSTM(input_dim=input_dim,
                                hidden_dim=args.hidden_dim,
                                n_scales=args.n_scales,
                                n_blocks_per_scale=args.blocks_per_scale,
                                num_classes=num_classes,
                                dropout=args.dropout)
    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    best_state = None

    for epoch in range(1, args.epochs + 1):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc, _, _ = eval_model(model, val_loader, criterion, device)
        print(f"Epoch {epoch:03d} | Train loss {train_loss:.4f} acc {train_acc*100:.2f}% | "
              f"Val loss {val_loss:.4f} acc {val_acc*100:.2f}%")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = model.state_dict().copy()
            torch.save({'model_state': best_state, 'label_encoder': le, 'feature_cols': feature_cols,
                        'scaler': scaler}, Path(args.out_dir) / 'best_model.pt')

    # Load best model
    if best_state is not None:
        model.load_state_dict(best_state)
    else:
        print("No improvement during training; using final weights.")

    # Evaluate on test
    test_loss, test_acc, y_true, y_pred = eval_model(model, test_loader, criterion, device)
    print(f"\nTest loss {test_loss:.4f} | Test Accuracy: {test_acc*100:.2f}%")

    # Decode labels
    y_true_dec = le.inverse_transform(y_true)
    y_pred_dec = le.inverse_transform(y_pred)

    print("\nClassification Report:")
    print(classification_report(y_true_dec, y_pred_dec, digits=4))

    # Confusion matrix
    cm = confusion_matrix(y_true_dec, y_pred_dec, labels=le.classes_)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_, cmap='Blues')
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix (Person recognition)")
    plt.tight_layout()
    plt.show()

    print("Saved best model to:", Path(args.out_dir) / 'best_model.pt')

import argparse
import sys

def get_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--data_path', type=str, default='combined_kinect_dataset.csv')
    parser.add_argument('--out_dir', type=str, default='.', help='directory to save model')
    parser.add_argument('--seq_len', type=int, default=64)
    parser.add_argument('--stride', type=int, default=16)
    parser.add_argument('--batch_size', type=int, default=64)
    parser.add_argument('--epochs', type=int, default=5)
    parser.add_argument('--lr', type=float, default=1e-3)
    parser.add_argument('--weight_decay', type=float, default=1e-5)
    parser.add_argument('--hidden_dim', type=int, default=64)
    parser.add_argument('--n_scales', type=int, default=3)
    parser.add_argument('--blocks_per_scale', type=int, default=2)
    parser.add_argument('--dropout', type=float, default=0.3)
    parser.add_argument('--seed', type=int, default=0)
    # splits: test_val_split is fraction that becomes temp; then test_ratio_of_temp splits test from temp
    parser.add_argument('--test_val_split', type=float, default=0.25)
    parser.add_argument('--test_ratio_of_temp', type=float, default=0.4)  # 0.4*0.25=0.10 overall test
    # prevents Jupyter from crashing
    if "ipykernel" in sys.argv[0]:
        return parser.parse_args([])
    else:
        return parser.parse_args()

args = get_args()
print(args)
main(args)


Namespace(batch_size=64, blocks_per_scale=2, data_path='combined_kinect_dataset.csv', dropout=0.3, epochs=5, hidden_dim=64, lr=0.001, n_scales=3, out_dir='.', seed=0, seq_len=64, stride=16, test_ratio_of_temp=0.4, test_val_split=0.25, weight_decay=1e-05)
Device: cpu
Loaded dataframe with shape: (129981, 79)
Detected feature columns count: 75
Persons (classes): 30
Class distribution: Counter({'Person_2': 4992, 'Person_6': 4938, 'Person_10': 4862, 'Person_26': 4426, 'Person_7': 4341, 'Person_14': 4298, 'Person_18': 4294, 'Person_8': 4292, 'Person_21': 4286, 'Person_20': 4282, 'Person_22': 4282, 'Person_12': 4275, 'Person_25': 4267, 'Person_9': 4267, 'Person_17': 4265, 'Person_16': 4264, 'Person_13': 4261, 'Person_3': 4258, 'Person_11': 4256, 'Person_19': 4252, 'Person_23': 4250, 'Person_4': 4246, 'Person_28': 4244, 'Person_5': 4243, 'Person_27': 4234, 'Person_29': 4234, 'Person_30': 4227, 'Person_1': 4215, 'Person_15': 4215, 'Person_24': 4215})
Windows: train 5546, val 1109, test 740
Epo

In [1]:
from time import sleep
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("TkAgg")

plt.ion()

# -----------------------------------------------------------
# Load model + metadata
# -----------------------------------------------------------
ckpt = torch.load("best_model.pt", map_location="cpu")

feature_cols = ckpt["feature_cols"]
scaler = ckpt["scaler"]
label_encoder = ckpt["label_encoder"]

input_dim = len(feature_cols)
num_classes = len(label_encoder.classes_)

model = MultiScaleGRSBiLSTM(
    input_dim=input_dim,
    hidden_dim=64,
    n_scales=3,
    n_blocks_per_scale=2,
    num_classes=num_classes,
    dropout=0.3
)
model.load_state_dict(ckpt["model_state"])
model.eval()

print("\nModel loaded successfully!\n")

# -----------------------------------------------------------
# Load dataset (NO SORTING)
# -----------------------------------------------------------
df = pd.read_csv("combined_kinect_dataset.csv")
df = df.dropna(subset=feature_cols + ["Person", "Activity", "timestamp"])

print("Available Activities:", df["Activity"].unique().tolist())
chosen_activity = input("Enter the activity to animate: ")

# row-wise filtering only
df_act = df[df["Activity"] == chosen_activity].reset_index(drop=True)

print(f"Total frames for '{chosen_activity}':", len(df_act))

# -----------------------------------------------------------
# Extract joint coordinates
# -----------------------------------------------------------
data = df_act[feature_cols].values.astype(np.float32)
scaled = scaler.transform(data)

num_joints = len(feature_cols) // 3

def get_xyz(row):
    coords = row.reshape(num_joints, 3)
    return coords[:, 0], coords[:, 1], coords[:, 2]

frames_xyz = [get_xyz(r) for r in data]

# -----------------------------------------------------------
# Skeleton bone connections (important!)
# -----------------------------------------------------------
KINECT_BONES = [
    (0,1), (1,20), (20,2), (2,3),
    (20,4), (4,5), (5,6),
    (20,8), (8,9), (9,10),
    (0,12), (12,13), (13,14),
    (0,16), (16,17), (17,18)
]

# -----------------------------------------------------------
# Prediction
# -----------------------------------------------------------
SEQ_LEN = 64

def predict_person(frame_idx):
    end = frame_idx + 1
    start = max(0, end - SEQ_LEN)

    window = scaled[start:end]

    if len(window) < SEQ_LEN:
        pad = SEQ_LEN - len(window)
        window = np.vstack([np.zeros((pad, window.shape[1])), window])

    # FIX: convert to float32 for LSTM
    x = torch.tensor(window, dtype=torch.float32).unsqueeze(0)

    with torch.no_grad():
        logits = model(x)
        pred = logits.argmax(dim=1).item()

    return label_encoder.inverse_transform([pred])[0]


# -----------------------------------------------------------
# LIVE ANIMATION
# -----------------------------------------------------------
fig = plt.figure(figsize=(7,7))
ax = fig.add_subplot(111, projection='3d')

# one line for each bone
bone_lines = [ax.plot([], [], [], 'o-', markersize=5)[0] for _ in KINECT_BONES]
title_text = ax.set_title("")

xs0, ys0, zs0 = frames_xyz[0]
ax.set_xlim(min(xs0)-0.5, max(xs0)+0.5)
ax.set_ylim(min(ys0)-0.5, max(ys0)+0.5)
ax.set_zlim(min(zs0)-0.5, max(zs0)+0.5)

for idx in range(len(frames_xyz)):
    x, y, z = frames_xyz[idx]

    # draw bones
    for line, (j1, j2) in zip(bone_lines, KINECT_BONES):
        line.set_data([x[j1], x[j2]], [y[j1], y[j2]])
        line.set_3d_properties([z[j1], z[j2]])

    pred_person = predict_person(idx)
    title_text.set_text(
        f"Activity: {chosen_activity} | Prediction: {pred_person}"
    )

    fig.canvas.draw()
    fig.canvas.flush_events()
    sleep(0.05)


NameError: name 'MultiScaleGRSBiLSTM' is not defined

In [ ]:
print(df_act.iloc[0][feature_cols].values[:30])
print(df_act.iloc[500][feature_cols].values[:30])


In [3]:
from time import sleep
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("TkAgg")

plt.ion()

# -----------------------------------------------------------
# Your Kinect Joints (EXACTLY AS GIVEN)
# -----------------------------------------------------------
joints = [
    "Head", "Neck", "SpineShoulder", "SpineMid", "SpineBase",
    "ShoulderLeft", "ElbowLeft", "WristLeft", "HandLeft",
    "ShoulderRight", "ElbowRight", "WristRight", "HandRight",
    "HipLeft", "KneeLeft", "AnkleLeft", "FootLeft",
    "HipRight", "KneeRight", "AnkleRight", "FootRight"
]

edges = [
    ("Head", "Neck"), ("Neck", "SpineShoulder"), ("SpineShoulder", "SpineMid"), ("SpineMid", "SpineBase"),
    ("SpineShoulder", "ShoulderLeft"), ("ShoulderLeft", "ElbowLeft"), ("ElbowLeft", "WristLeft"), ("WristLeft", "HandLeft"),
    ("SpineShoulder", "ShoulderRight"), ("ShoulderRight", "ElbowRight"), ("ElbowRight", "WristRight"), ("WristRight", "HandRight"),
    ("SpineBase", "HipLeft"), ("HipLeft", "KneeLeft"), ("KneeLeft", "AnkleLeft"), ("AnkleLeft", "FootLeft"),
    ("SpineBase", "HipRight"), ("HipRight", "KneeRight"), ("KneeRight", "AnkleRight"), ("AnkleRight", "FootRight")
]

# Map joint names → indices
joint_index = {j:i for i,j in enumerate(joints)}

# convert edges into index pairs
KINECT_BONES = [(joint_index[a], joint_index[b]) for a, b in edges]

# -----------------------------------------------------------
# Load model + metadata
# -----------------------------------------------------------
ckpt = torch.load("best_model.pt", map_location="cpu")

feature_cols = ckpt["feature_cols"]       # must match your dataset joint layout
scaler = ckpt["scaler"]
label_encoder = ckpt["label_encoder"]

input_dim = len(feature_cols)
num_classes = len(label_encoder.classes_)

model = MultiScaleGRSBiLSTM(
    input_dim=input_dim,
    hidden_dim=64,
    n_scales=3,
    n_blocks_per_scale=2,
    num_classes=num_classes,
    dropout=0.3
)
model.load_state_dict(ckpt["model_state"])
model.eval()

print("\nModel loaded successfully!\n")

# -----------------------------------------------------------
# Load dataset
# -----------------------------------------------------------
df = pd.read_csv("combined_kinect_dataset.csv")
df = df.dropna(subset=feature_cols + ["Person", "Activity", "timestamp"])

print("Available Activities:", df["Activity"].unique().tolist())
chosen_activity = input("Enter the activity to animate: ")

df_act = df[df["Activity"] == chosen_activity].reset_index(drop=True)
print(f"Total frames for '{chosen_activity}':", len(df_act))

# -----------------------------------------------------------
# Extract skeleton joint coordinates (X,Y,Z)
# -----------------------------------------------------------
num_joints = len(joints)

def extract_xyz(row):
    xs, ys, zs = [], [], []
    for j in joints:
        xs.append(row[f"{j}_x"])
        ys.append(row[f"{j}_y"])
        zs.append(row[f"{j}_z"])
    return np.array(xs), np.array(ys), np.array(zs)

frames_xyz = [extract_xyz(row) for _, row in df_act.iterrows()]

# also prepare scaled features for prediction window
data = df_act[feature_cols].values.astype(np.float32)
scaled = scaler.transform(data)

# -----------------------------------------------------------
# Prediction
# -----------------------------------------------------------
SEQ_LEN = 64

def predict_person(frame_idx):
    end = frame_idx + 1
    start = max(0, end - SEQ_LEN)

    window = scaled[start:end]

    if len(window) < SEQ_LEN:
        pad = SEQ_LEN - len(window)
        window = np.vstack([np.zeros((pad, window.shape[1])), window])

    x = torch.tensor(window, dtype=torch.float32).unsqueeze(0)

    with torch.no_grad():
        logits = model(x)
        pred = logits.argmax(dim=1).item()

    return label_encoder.inverse_transform([pred])[0]

# -----------------------------------------------------------
# LIVE ANIMATION
# -----------------------------------------------------------
fig = plt.figure(figsize=(7,7))
ax = fig.add_subplot(111, projection='3d')

bone_lines = [ax.plot([], [], [], 'o-', markersize=5)[0] for _ in KINECT_BONES]
title_text = ax.set_title("")

xs0, ys0, zs0 = frames_xyz[0]
ax.set_xlim(min(xs0)-0.5, max(xs0)+0.5)
ax.set_ylim(min(ys0)-0.5, max(ys0)+0.5)
ax.set_zlim(min(zs0)-0.5, max(zs0)+0.5)

for idx, (x, y, z) in enumerate(frames_xyz):

    # Draw all bones
    for line, (j1, j2) in zip(bone_lines, KINECT_BONES):
        line.set_data([x[j1], x[j2]], [y[j1], y[j2]])
        line.set_3d_properties([z[j1], z[j2]])

    pred_person = predict_person(idx)
    title_text.set_text(
        f"Activity: {chosen_activity} | Prediction: {pred_person}"
    )

    fig.canvas.draw()
    fig.canvas.flush_events()
    sleep(0.05)



Model loaded successfully!

Available Activities: ['bending', 'jumping', 'running', 'sitting', 'squat', 'standing', 'walking']
Total frames for 'jumping': 18780


: 